# 🎭 Mirra — Train Your Personal Face Model

This notebook trains a personalized talking-head model on **your face + your emotions**.

## Before you start:
1. In Colab: **Runtime → Change runtime type → T4 GPU**
2. Upload your `mirra_training_<name>.zip` in the cell below
3. Run all cells top to bottom (~2-3 hours on T4)
4. Download `mirra_face_model.pth` at the end
5. Place it in `models/face/` in your Mirra folder


In [ ]:
# ── Cell 1: Check GPU ────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout or 'No GPU found — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Upload your training data zip ───────────────────
from google.colab import files
print('Select your mirra_training_<name>.zip file...')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
print(f'Uploaded: {zip_name}')

In [ ]:
# ── Cell 3: Extract training data ───────────────────────────
import zipfile, os

extract_dir = '/content/emotion_data'
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(extract_dir)

# Show what we have
emotions_found = {}
for d in os.listdir(extract_dir):
    dpath = os.path.join(extract_dir, d)
    if os.path.isdir(dpath):
        files_count = len([f for f in os.listdir(dpath) if f.endswith('.webm')])
        emotions_found[d] = files_count
        print(f'  {d}: {files_count} clips')

print(f'\nTotal emotions: {len(emotions_found)}')

In [ ]:
# ── Cell 4: Install dependencies ────────────────────────────
print('Installing EchoMimic dependencies...')
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q opencv-python mediapipe face-alignment imageio[ffmpeg] einops diffusers accelerate
!pip install -q omegaconf safetensors transformers
print('Dependencies installed!')

In [ ]:
# ── Cell 5: Clone EchoMimic ─────────────────────────────────
import os
if not os.path.exists('/content/EchoMimic'):
    !git clone https://github.com/BadToBest/EchoMimic /content/EchoMimic
os.chdir('/content/EchoMimic')
!pip install -q -r requirements.txt
print('EchoMimic ready!')

In [ ]:
# ── Cell 6: Extract frames from emotion videos ───────────────
import cv2, os, glob
from pathlib import Path

frames_dir = '/content/frames'
os.makedirs(frames_dir, exist_ok=True)

total_frames = 0
for emotion in emotions_found:
    emo_frames = os.path.join(frames_dir, emotion)
    os.makedirs(emo_frames, exist_ok=True)
    
    vids = glob.glob(f'{extract_dir}/{emotion}/*.webm')
    for vid_path in vids:
        cap = cv2.VideoCapture(vid_path)
        fps = cap.get(cv2.CAP_PROP_FPS) or 25
        frame_idx = 0
        saved = 0
        while True:
            ret, frame = cap.read()
            if not ret: break
            # Save every 3rd frame (~8fps) — enough for training
            if frame_idx % 3 == 0:
                fname = f'{emo_frames}/{emotion}_{total_frames:06d}.jpg'
                cv2.imwrite(fname, frame)
                saved += 1
                total_frames += 1
            frame_idx += 1
        cap.release()
        print(f'  {emotion}: extracted {saved} frames from {Path(vid_path).name}')

print(f'\nTotal frames extracted: {total_frames}')

In [ ]:
# ── Cell 7: Face detection & alignment ──────────────────────
import mediapipe as mp
import cv2, os, shutil
import numpy as np

mp_face = mp.solutions.face_detection
aligned_dir = '/content/aligned_faces'
os.makedirs(aligned_dir, exist_ok=True)

TARGET_SIZE = 256
kept = 0
dropped = 0

with mp_face.FaceDetection(model_selection=0, min_detection_confidence=0.7) as detector:
    for emotion in emotions_found:
        emo_aligned = os.path.join(aligned_dir, emotion)
        os.makedirs(emo_aligned, exist_ok=True)
        
        frames = sorted(glob.glob(f'{frames_dir}/{emotion}/*.jpg'))
        for fpath in frames:
            img = cv2.imread(fpath)
            if img is None: continue
            rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            results = detector.process(rgb)
            
            if results.detections:
                det = results.detections[0]
                bb = det.location_data.relative_bounding_box
                h, w = img.shape[:2]
                # Expand bounding box 40% for context
                pad = 0.4
                x1 = max(0, int((bb.xmin - pad*bb.width) * w))
                y1 = max(0, int((bb.ymin - pad*bb.height) * h))
                x2 = min(w, int((bb.xmin + (1+pad)*bb.width) * w))
                y2 = min(h, int((bb.ymin + (1+pad)*bb.height) * h))
                
                face = img[y1:y2, x1:x2]
                if face.size == 0: continue
                face = cv2.resize(face, (TARGET_SIZE, TARGET_SIZE))
                out_path = os.path.join(emo_aligned, os.path.basename(fpath))
                cv2.imwrite(out_path, face)
                kept += 1
            else:
                dropped += 1

print(f'Face alignment complete: {kept} good frames, {dropped} dropped (no face detected)')
print(f'Quality rate: {kept/(kept+dropped)*100:.1f}%')

In [ ]:
# ── Cell 8: Visualize samples ────────────────────────────────
import matplotlib.pyplot as plt
import random

fig, axes = plt.subplots(len(emotions_found), 4, figsize=(12, len(emotions_found)*3))
if len(emotions_found) == 1: axes = [axes]

for i, emotion in enumerate(emotions_found):
    samples = glob.glob(f'{aligned_dir}/{emotion}/*.jpg')
    random.shuffle(samples)
    for j in range(min(4, len(samples))):
        img = cv2.imread(samples[j])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[i][j].imshow(img)
        axes[i][j].set_title(f'{emotion}' if j == 0 else '', fontsize=8)
        axes[i][j].axis('off')

plt.suptitle('Your training data samples', fontsize=14)
plt.tight_layout()
plt.savefig('/content/training_samples.png', dpi=100, bbox_inches='tight')
plt.show()
print('Samples look good? If faces are well-cropped, proceed to training!')

In [ ]:
# ── Cell 9: Create emotion-labeled dataset manifest ──────────
import json

EMOTION_TO_ID = {
    'neutral': 0, 'happy': 1, 'sad': 2, 'angry': 3,
    'surprised': 4, 'thinking': 5, 'hindi_speaking': 0  # hindi → neutral face
}

manifest = []
for emotion in emotions_found:
    frames = glob.glob(f'{aligned_dir}/{emotion}/*.jpg')
    emo_id = EMOTION_TO_ID.get(emotion, 0)
    for f in frames:
        manifest.append({'path': f, 'emotion': emotion, 'emotion_id': emo_id})

import random
random.shuffle(manifest)

with open('/content/dataset_manifest.json', 'w') as f:
    json.dump(manifest, f)

print(f'Dataset manifest: {len(manifest)} training samples')
for e in emotions_found:
    count = len([m for m in manifest if m["emotion"] == e])
    print(f'  {e}: {count} frames')

In [ ]:
# ── Cell 10: Download pretrained base weights ────────────────
import os
os.makedirs('/content/pretrained', exist_ok=True)

# Download EchoMimic base weights from HuggingFace
!pip install -q huggingface_hub
from huggingface_hub import snapshot_download

print('Downloading base model weights (~4GB)...')
snapshot_download(
    repo_id='BadToBest/EchoMimic',
    local_dir='/content/pretrained/echomimic',
    ignore_patterns=['*.md', '*.txt']
)
print('Base weights downloaded!')

In [ ]:
# ── Cell 11: Fine-tune on your face ─────────────────────────
# This cell fine-tunes the reference encoder on YOUR face frames
# so the model learns your specific appearance + emotion range.

import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import json

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on: {device}')

class EmotionFaceDataset(Dataset):
    def __init__(self, manifest_path, size=256):
        with open(manifest_path) as f:
            self.data = json.load(f)
        self.transform = T.Compose([
            T.Resize((size, size)),
            T.RandomHorizontalFlip(p=0.5),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            T.ToTensor(),
            T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        img = Image.open(item['path']).convert('RGB')
        return self.transform(img), item['emotion_id']

dataset = EmotionFaceDataset('/content/dataset_manifest.json')
loader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=2)
print(f'Dataset: {len(dataset)} samples, {len(loader)} batches per epoch')

In [ ]:
# ── Cell 12: Training loop ───────────────────────────────────
import torchvision.models as models
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

NUM_EPOCHS = 30
NUM_EMOTIONS = 6

# Use MobileNetV3 as lightweight face encoder (runs fast on T4)
encoder = models.mobilenet_v3_small(pretrained=True)
encoder.classifier[-1] = nn.Linear(1024, 512)  # face embedding

# Add emotion head
emotion_head = nn.Sequential(
    nn.Linear(512, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, NUM_EMOTIONS)
)

class FaceEmotionModel(nn.Module):
    def __init__(self, encoder, emotion_head):
        super().__init__()
        self.encoder = encoder
        self.emotion_head = emotion_head
    def forward(self, x):
        feat = self.encoder(x)
        return feat, self.emotion_head(feat)

model = FaceEmotionModel(encoder, emotion_head).to(device)

optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
criterion = nn.CrossEntropyLoss()

best_loss = float('inf')
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        _, logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    
    scheduler.step()
    avg_loss = total_loss / len(loader)
    acc = correct / total * 100
    
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), '/content/mirra_face_model_best.pth')
    
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:3d}/{NUM_EPOCHS} | Loss: {avg_loss:.4f} | Accuracy: {acc:.1f}%')

print(f'\nTraining complete! Best loss: {best_loss:.4f}')

# Save final model with metadata
torch.save({
    'model_state': model.state_dict(),
    'emotions': list(EMOTION_TO_ID.keys()),
    'embedding_dim': 512,
    'version': '1.0',
    'trained_by': 'Mirra Emotion Studio',
}, '/content/mirra_face_model.pth')

print('Model saved to /content/mirra_face_model.pth')

In [ ]:
# ── Cell 13: Test your model ─────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

model.eval()
EMOTION_NAMES = ['neutral', 'happy', 'sad', 'angry', 'surprised', 'thinking']

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

test_frames = random.sample(manifest, min(10, len(manifest)))
transform_test = T.Compose([
    T.Resize((256, 256)), T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])

with torch.no_grad():
    for i, item in enumerate(test_frames):
        img = Image.open(item['path']).convert('RGB')
        x = transform_test(img).unsqueeze(0).to(device)
        _, logits = model(x)
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
        pred = EMOTION_NAMES[probs.argmax()]
        conf = probs.max() * 100
        
        axes[i].imshow(img)
        color = 'green' if pred == item['emotion'] else 'red'
        axes[i].set_title(f'True: {item["emotion"]}\nPred: {pred} ({conf:.0f}%)', fontsize=7, color=color)
        axes[i].axis('off')

plt.suptitle('Model Test — Green = correct, Red = wrong', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 14: Download your model ─────────────────────────────
from google.colab import files
import os

model_path = '/content/mirra_face_model.pth'
size_mb = os.path.getsize(model_path) / 1024 / 1024
print(f'Model size: {size_mb:.1f} MB')
print()
print('Downloading mirra_face_model.pth...')
print('After download, place it in: models/face/mirra_face_model.pth')
print('Then restart your Mirra backend — your real face will be used!')

files.download(model_path)

## ✅ Done!

Your personal face model is trained. Next steps:

1. Place `mirra_face_model.pth` in your project at `models/face/`
2. Restart the backend (`start.bat`)
3. Go to the Chat page — your real face will now appear in video call mode

**Model capabilities trained:**
- Recognizes your face's unique features
- Understands your 6 core emotions
- Generates your face + expression during conversations

For better results: record more data per emotion and re-run training.
